# The core maths notebook
- Implementation of get_coords()
- Implementation of bivariate_k()

### Core concepts:
- K function includes parameters for:
    - Point pattern A (n_A points)
    - Point pattern B (n_B points)
    - Window area (|W|)
    - w_ij is an edge correction weight

Under CSR, K_AB (r) = πr^2

- The L-function is the vairance-stabilised transform
    L_AB(r) = sqrt(K_AB(r)/π) - r

Under CSR, L(r) = 0
Values above 0 indicate co-localisation at scale r

- Edge detection
    - Without it, points near the boundaries appear to have fewer neighbours than they really do
    - Simplest correction is to weight each pair by the inverse of the fraction of the circle centered at i with radius d_ij that falls inside W
        - e.g. if only 1/2 of the radius is inside W, multiply counts by 2

## Import dependencies and data

In [48]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

fov3 = pd.read_parquet("../data/processed/fov3_strips.parquet")

# Gene pair: KRT8 x KRT18
# Chosen because they are obligate heterodimer partners in simple epithelial
# cells — strong prior expectation of co-localisation. This is our positive
# control: if the K-function doesn't detect co-localisation here, the
# implementation is wrong.
GENE_A = 'KRT8'
GENE_B = 'KRT18'

## The core functions

### get_coords()
- Retrieves coordinates of points for a given gene

In [ ]:
def get_coords(df: pd.DataFrame, 
gene: str,
x_col: str = 'x_global_px',
y_col: str = 'y_global_px') -> np.ndarray:
    """Extract coordinates for a specific gene.
     Parameters:
     df : pd.DataFrame
         The dataframe containing the data. Must contain columns: x_col, y_col, and 'target'.
    gene : str
        Gene nams for coordinate extraction. Defaults match CosMx global coordiantes.
    

    Usage:
    get_coords(tissue_df, gene="KRT8")

    Returns:
    np.ndarray, shape (N, 2)
        An array of x and y coordinates for the specified gene. Returns (0,2) array if gene is absent.
     """
    coords = df.loc[df['target'] == gene, [x_col, y_col]].values
    return coords.astype(np.float64)

### get_window()
- Derives the bounding box window from a transcript dataframe


In [50]:
def get_window(df: pd.DataFrame,
                x_col: str = 'x_global_px',
                y_col: str = 'y_global_px') -> tuple:
    """
    Derive the counding box window from a transcript DataFrame.

    Usage:
    get_window(tissue_df)

    Returns:
    tuple: (x_min, x_max, y_min, y_max)
    """
    return (
        df[x_col].min(), df[x_col].max(),
        df[y_col].min(), df[y_col].max()
    )

### bivariate_k()
- Estimate the bivariate (cross-type) Ripley's K-function K_AB(r)

In [49]:
def bivariate_k(coords_a: np.ndarray,
                coords_b: np.ndarray,
                r_vals: np.ndarray,
                window: tuple) -> np.ndarray:
    """
    Estimate the biavariate (cross-type) Rispleys K-function K_AB(r).

    For each radius r, counts how many B-transcripts fall within distance r 
    of each A-transcript, applies Ripley's isotropic edge correction, and 
    normalised by the expected count under CSR.

    Under CSR: K_AB(r) = πr^2.

    Uses scipu.spatial.cKDTree for 0(n log n) neighbour counting.
    Can't just use naive distance matrix (0(n^2), because of scaling issues.

    Parameters:
    coords_a : np.ndarray, shape (n_a, 2)
    coords_b : np.ndarray, shape (n_b, 2)
    r_vals : np.ndarray, shape(n_r,)
        Radii to evaluate. Keep r_max < half the window's narrowest dimension.
        (Avoids severe edge correction instability)
    window : tuple
        (x_min, x_max, y_min, y_max). 
        The K-function assumes stationarity within the window.

    Usage:
    bivariate_k(coords_a, coords_b, r_vals, window)

    Returns:
    np.ndarray, shape (n_r,)
        Estimated K_AB(r) at each radius.
    """
    n_a, n_b = len(coords_a), len(coords_b)
    if n_a < 10 or n_b < 10:
        raise ValueError(f"Too few points: n_a={n_a}, n_b={n_b}. Minimum 10.")

    x_min, x_max, y_min, y_max = window
    area = (x_max - x_min) * (y_max - y_min)
    lambda_b = n_b / area  # intensity of pattern B (points per px^2)

    tree_b = cKDTree(coords_b)
    k_vals = np.zeros(len(r_vals))

    for i, r in enumerate(r_vals):
        if r == 0:
            continue

        neighbours = tree_b.query_ball_point(coords_a, r=r)
        counts = np.array([len(nb) for nb in neighbours], dtype=np.float64)

        # Ripley's isotropic edge correction:
        # Points near the window boundary have part of their search circle
        # outside the observed region. We estimate the fraction of the circle
        # that lies inside and upweight by its inverse.
        # Derivation: distance from each point to each edge; if edge distance
        # < r, that edge clips the circle. arccos(d/r) gives the clipped angle.
        dx_left  = coords_a[:, 0] - x_min
        dx_right = x_max - coords_a[:, 0]
        dy_bot   = coords_a[:, 1] - y_min
        dy_top   = y_max - coords_a[:, 1]

        # Fraction of circle inside window (sum of inside arc fractions per edge)
        # Each term is in [0, 0.5]; sum of all four is in (0, 2.0]
        def arc_in(d):
            return np.arccos(np.clip(d / r, 0, 1)) / np.pi

        inside = (arc_in(dx_left) + arc_in(dx_right) +
                  arc_in(dy_bot)  + arc_in(dy_top))
        # Clamp to avoid division by very small weights at corners
        inside = np.clip(inside, 0.1, 1.0)

        k_vals[i] = (counts / inside).sum() / (n_a * lambda_b)

    return k_vals

### k_to_l()
- Transforms K(r) to variance-stabilised L(r)

In [51]:
def k_to_l(k_vals: np.ndarray,
           r_vals: np.ndarray) -> np.ndarray:
    """            
    Transform K-function to variance-stabilised L-function.

    L(r) - sqrt(K(r) / π) - r

    Under CSR, L(r) = 0.
    L(r) > 0 : co-localisation at scale r.
    L(r) < 0 : spatial inhibition / repulsion at scale r.    
    """
    return np.sqrt(np.maximum(k_vals, 0) / np.pi) - r_vals

### compute_envelope()
- Monte Carlo permutation envelope for the bivariate L-function


In [52]:
def compute_envelope(coords_a: np.ndarray,
                     coords_b: np.ndarray,
                     r_vals: np.ndarray,
                     window: tuple,
                     n_sim: int = 99,
                     seed: int = 42) -> tuple:
    """
    Monte Carlo permutation envelope for the bivariate L-function.

    Null model: random permutation. Pool all coords_a and coords_b,
    randomly reassign n_a points to pattern A and n_b points to pattern B,
    recompute L(r). Repeat n_sim times to build a pointwise envelope.

    This null preserves total transcript intensity and spatial distrivbution
    while destroying any association *between* the two patterns.

    Parameters:
    n_sim : int
        99 simulations -> pointwise alhpa ~0.02 (observed exceeds all sims).
        Use 999 for publication figures.
    seed : int
        Fixed for reproducibility. Record in methods.

    Usage:
    l_lo, l_hi = compute_envelope(coords_a, coords_b, r_vals, window)

    Returns:
    l_lo, l_hi : np.ndarray, shape (n_r,)
        Pointwise min and max of simulation L(r) across all n_sim runs.
    """
    rng = np.random.default_rng(seed)
    n_a = len(coords_a)
    pooled = np.vstack([coords_a, coords_b])
    
    sim_l  = np.zeros((n_sim, len(r_vals)))

    for s in range(n_sim):
        idx = rng.permutation(len(pooled))
        sim_a = pooled[idx[:n_a]]
        sim_b = pooled[idx[n_a:]]
        k_sim = bivariate_k(sim_a, sim_b, r_vals, window)
        sim_l[s] = k_to_l(k_sim, r_vals)

    return sim_l.min(axis=0), sim_l.max(axis=0)

In [55]:
coords_a = get_coords(fov3, gene=GENE_A)
coords_b = get_coords(fov3, gene=GENE_B)

window = get_window(fov3)

r_vals = np.linspace(0, 300, 60)
k_obs = bivariate_k(coords_a, coords_b, r_vals, window)
print(k_obs)
print(f"bivariate_k output shape: {k_obs.shape}")   # expect (60,)
print(f"K(r=0)   = {k_obs[0]:.2f}")                 # expect (0)

# Check against CSR theoretical at r=100 and r=200
for r_check in [100, 200]:
    idx = np.argmin(np.abs(r_vals - r_check))
    csr = np.pi * r_check**2
    print(f"K(r={r_check}) = {k_obs[idx]:.0f}  |  CSR = {csr:.0f}  |  ratio = {k_obs[idx]/csr:.2f}")
# ratio > 1 means more clustering than CSR — expected for KRT8/KRT18

l_obs = k_to_l(k_obs, r_vals)   
print(f"k_to_l output shape: {l_obs.shape}")   # expect (60,)
print(f"L(r=0)   = {l_obs[0]:.3f}")            # expect (0)

# Verify the transform manually at one point
idx_100 = np.argmin(np.abs(r_vals - 100))
manual  = np.sqrt(k_obs[idx_100] / np.pi) - 100
print(f"L(r=100) = {l_obs[idx_100]:.3f}  |  manual check = {manual:.3f}") # expect ~identical

# Synthetic CSR sanity check: if K = pi*r^2 exactly, L should = 0
k_csr   = np.pi * r_vals**2
l_csr   = k_to_l(k_csr, r_vals)
print(f"Max |L| for perfect CSR input: {np.abs(l_csr).max():.6f}") # expect ~0

l_lo, l_hi = compute_envelope(coords_a, coords_b, r_vals, window, 9)
print(f"Envelope shapes: l_lo={l_lo.shape}, l_hi={l_hi.shape}") # expect (60,) (60,)

# The envelope should straddle zero (simulations are under the null)
print(f"Envelope lo at r=100: {l_lo[np.argmin(np.abs(r_vals-100))]:.2f}")
print(f"Envelope hi at r=100: {l_hi[np.argmin(np.abs(r_vals-100))]:.2f}")
# Observed should sit above hi if there is genuine co-localisation
print(f"Observed  L at r=100: {l_obs[np.argmin(np.abs(r_vals-100))]:.2f}")

[      0.           13080.06212272   47088.2236418    91560.43485905
  183120.8697181   285145.35427533  367864.51475766  480344.00342952
  588282.27386612  736055.98267958  849829.99910022  975841.82210366
 1091533.89944013 1206127.30480601 1316448.17153136 1462182.09238978
 1597369.12973389 1716800.61658781 1820119.96255191 1963170.47299389
 2111120.53943668 2255186.90091599 2376928.20261006 2544152.90627307
 2710766.83391136 2857873.88073963 2976817.56730796 3108334.56081339
 3271479.81993785 3418986.44487023 3555023.73113268 3688241.15840125
 3831442.06319075 4003186.68726262 4138414.56983996 4281973.83565937
 4386800.17862996 4531815.83812643 4667304.09675359 4787848.45783011
 4919569.33677708 5049077.23615575 5161968.42385634 5269148.51781452
 5382341.88630449 5505007.08221026 5631912.55671665 5752141.74048276
 5869359.39438515 5996134.83712013 6111626.15496958 6193613.7414591
 6306557.57182167 6382988.82832824 6494626.65551191 6613023.65131001
 6702567.14382884 6778302.42694584 

In [53]:
STRIP = 'strip_2'
strip_df = fov3[fov3['strip'] == STRIP]

coords_a = get_coords(strip_df, GENE_A)
coords_b = get_coords(strip_df, GENE_B)
window   = get_window(strip_df)

print(f"{GENE_A}: {len(coords_a)} transcripts")
print(f"{GENE_B}: {len(coords_b)} transcripts")
print(f"Window: x=[{window[0]:.0f}, {window[1]:.0f}], "
      f"y=[{window[2]:.0f}, {window[3]:.0f}]")

narrow_dim = min(window[1]-window[0], window[3]-window[2])
print(f"Narrowest window dimension: {narrow_dim:.0f} px")
print(f"Recommended r_max: < {narrow_dim/2:.0f} px")

KRT8: 74 transcripts
KRT18: 118 transcripts
Window: x=[13052, 14252], y=[34219, 38450]
Narrowest window dimension: 1200 px
Recommended r_max: < 600 px


In [54]:
# 60 points gives smooth curve without excessive computation
r_vals = np.linspace(0, 300, 60)

k_obs = bivariate_k(coords_a, coords_b, r_vals, window)
l_obs = k_to_l(k_obs, r_vals)

# Spot-check before running the expensive envelope
print("L(r) at key scales:")
for r_check in [50, 100, 150, 200]:
    idx = np.argmin(np.abs(r_vals - r_check))
    print(f"  r={r_check}px: L={l_obs[idx]:.2f}  (CSR expectation: 0)")

L(r) at key scales:
  r=50px: L=523.12  (CSR expectation: 0)
  r=100px: L=786.41  (CSR expectation: 0)
  r=150px: L=978.65  (CSR expectation: 0)
  r=200px: L=1084.55  (CSR expectation: 0)
